# Lecture 09 — Practice Exercises

All exercises from Lecture 09 collected in one place. Work through them after studying the corresponding lecture material.

Each exercise is tagged with **the learning goal it covers** (see `reading_material/goals_09.md`) and **its difficulty tier**:

- `[G1]`–`[G8]`: required goals (every required exercise must be completed)
- `[O1]`–`[O3]`: optional / career-track goals (the stretch exercises only)
- *trivial* ≈ 15 min, *realistic* ≈ 30–45 min, *stretch* ≈ 90 min

| Section | Source notebook | Topic |
|---|---|---|
| 0 | `lec_09a` | Concept warm-up: unsupervised vs supervised |
| 1 | `lec_09a` | KMeans on the Mall Customers dataset |
| 2 | `lec_09a` | Choosing `K` — Elbow + Silhouette + cluster profiling |
| 3 | `lec_09b` | KMeans assumptions and where it breaks |
| 4 | `lec_09c` | Wrap a trained KMeans in a Gradio app |
| 5 | `lec_09d` *(optional)* | Tuning `init` and `n_init` (stretch) |

Solutions for every required exercise (and the stretch one) are in `lec_09_exercises_solutions.ipynb`.


---
## Setup — Load libraries and data

Run this cell first to have everything ready for the exercises below. The `mall_customers.csv` file lives in the `reading_material/` folder next door, so we load it from there with a relative path.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.datasets import make_moons, make_blobs


## 0. Setup paths to the dataset on your computer.

In [ ]:
# Mall customers dataset (used in sections 1, 2 and 4)

DATA_PATH = '../reading_material/mall_customers.csv'   # adjust if your layout differs
df = pd.read_csv(DATA_PATH)

# Drop the useless ID column
df = df.drop(columns=["CustomerID"])

print(f"Mall customers dataset: {df.shape[0]} rows, {df.shape[1]} columns")
df.head(3)


---
### Exercise 0.1 [G1, *trivial*] — Unsupervised vs supervised

In a markdown cell or comment block below, answer in **one sentence**:

> *In one sentence, what is the difference between supervised and unsupervised learning, and name one task each one is typically used for.*

This is a concept-only exercise — no code required. Aim for one clear sentence; if you need three sentences, your distinction isn't tight enough yet.


In [ ]:
# Exercise 0.1 — Your one-sentence answer here, as a comment or in a markdown cell.

# Supervised learning ...
# Unsupervised learning ...


---
## 1. KMeans on the Mall Customers dataset

*From `lec_09a_kmeans_clustering.ipynb`*

In the lecture we walked through the full KMeans workflow on this dataset: load → EDA → fit → predict → evaluate → label clusters. Now you do it yourself, with a slightly different setup each time.


### Exercise 1.1 [G3, *realistic*] — Quick EDA

Before fitting any model, get a feel for the data.

**Steps:**
1. Print the column names and data types.
2. Print descriptive statistics for the **numerical** columns.
3. Print the count (and percentage) of each value in `Genre`.
4. Build a `sns.pairplot()` of the numerical columns, coloured by `Genre`.

**Question to answer in a markdown cell below your code:** Looking at the pairplot, how many natural groupings do you think you see? Use that intuition to set the `K` you will try later.


In [ ]:
# Exercise 1.1 — Your code here


### Exercise 1.2 [G2 + G3, *realistic*] — KMeans with two features and `K = 5`

Fit a `KMeans` model on the two features `Annual_Income_(k$)` and `Spending_Score`, asking for **5 clusters**.

**Steps:**
1. Build the feature matrix `X` with these two columns.
2. Create `model = KMeans(n_clusters=5, random_state=42, n_init=10)`.
3. Use `fit_predict(X)` to get cluster labels in **one step**.
4. Add the labels as a new column on `df` named `cluster`.
5. Inspect `model.cluster_centers_`, `model.labels_`, and `model.inertia_`. Print all three.
6. Make a 2D scatter plot coloured by `cluster`, with the centroids overlaid.


In [ ]:
# Exercise 1.2 — Your code here


### Exercise 1.3 [G6, *trivial*] — Predict the cluster of a new customer

Imagine three new customers walk into the mall:

| Customer | Annual income (k$) | Spending score |
|---|---|---|
| A | 20  | 80 |
| B | 75  | 50 |
| C | 110 | 15 |

Using the model you trained in Exercise 1.2, predict which cluster each one belongs to.

**Steps:**
1. Build a small `pandas.DataFrame` with the three new customers (use the same column names as the training data).
2. Call `model.predict()` on the new DataFrame.
3. Print the cluster id for each customer.

**Question to answer in a markdown cell below:** Why is this called *predict* even though there is no `y` target?


In [ ]:
# Exercise 1.3 — Your code here


---
## 2. Choosing `K` — Elbow + Silhouette + profiling

*From `lec_09a_kmeans_clustering.ipynb`*

`K` is a hyperparameter you must pick yourself. We use two metrics in the lecture: **WCSS** (within-cluster sum of squares, the "elbow") and the **silhouette score**.


### Exercise 2.1 [G4, *realistic*] — Elbow plot from K = 1 to K = 10

Compute the WCSS for `K = 1, 2, ..., 10` on the **two-feature** matrix (`Annual_Income_(k$)`, `Spending_Score`) and plot it.

**Steps:**
1. Loop `k` from 1 to 10, fit a fresh `KMeans` for each, and append `model.inertia_` to a list.
2. Plot `K` on the x-axis and WCSS on the y-axis.
3. Mark with a vertical dashed line the `K` that you think is the elbow.


In [ ]:
# Exercise 2.1 — Your code here


### Exercise 2.2 [G4, *realistic*] — Silhouette score from K = 2 to K = 10

The silhouette score is **not defined for K = 1** (you need at least two clusters). Compute it for `K = 2, 3, ..., 10` on the same two features and plot it.

**Steps:**
1. Loop `k` from 2 to 10, fit `KMeans`, predict labels, and compute `silhouette_score(X, labels)`.
2. Plot `K` vs silhouette score.
3. Which `K` gives the **highest** score? Does it agree with your elbow choice from 2.1?


In [ ]:
# Exercise 2.2 — Your code here


### Exercise 2.3 [G5, *realistic*] — Apply your chosen `K` with **four** features and profile the clusters

Now use **all four** features: `Genre`, `Age`, `Annual_Income_(k$)`, `Spending_Score`.

**Steps:**
1. Encode `Genre` as a numeric `Gender` column (`Male` → 1, `Female` → 0) so KMeans can use it.
2. Build `X` with the four numeric columns.
3. Fit `KMeans` with the `K` you chose in 2.1 / 2.2 (use `random_state=42`, `n_init=10`).
4. Print the per-cluster mean of every numeric feature (`df.groupby('cluster').mean()`).
5. Give each cluster a one- or two-word **human-readable label** (e.g. "young high-spenders") in a small dictionary `cluster_labels = {0: "...", 1: "...", ...}`.
6. Add a `cluster_label` column to `df` and re-plot the 2D scatter using the labels in the legend.


In [ ]:
# Exercise 2.3 — Your code here


---
## 3. KMeans assumptions and where it breaks

*From `lec_09b_kmeans_assumptions_caveats.ipynb`*

KMeans implicitly assumes that clusters are **roughly spherical**, **of similar size**, and **of similar variance**. When those assumptions are violated, the algorithm produces clusters that look obviously wrong to the human eye. These exercises reproduce two of those failure modes on synthetic data.


### Exercise 3.1 [G7, *realistic*] — KMeans fails on non-convex shapes (`make_moons`)

Generate a small "two moons" dataset and try to cluster it with KMeans.

**Steps:**
1. `from sklearn.datasets import make_moons` and create `X, y = make_moons(n_samples=400, noise=0.06, random_state=42)`.
2. Plot `X` coloured by the true label `y` — you should see two interlocking moons.
3. Fit `KMeans(n_clusters=2, random_state=42, n_init=10)` on `X`, predict labels, and plot `X` coloured by **the KMeans labels** next to the true-label plot.

**Question to answer in a markdown cell below:** Why does KMeans cut the moons across instead of along their shape? What property of the data does it violate?


In [ ]:
# Exercise 3.1 — Your code here


### Exercise 3.2 [G7, *realistic*] — KMeans struggles with anisotropic (stretched) blobs

Same idea, different failure mode: clusters that are **stretched along diagonals** instead of being round.

**Steps:**
1. Build three blobs with `make_blobs(n_samples=600, centers=3, cluster_std=0.6, random_state=42)`.
2. Apply an anisotropic linear transformation to make the blobs diagonal, e.g. `X_aniso = X @ np.array([[0.6, -0.6], [-0.4, 0.8]])`.
3. Fit `KMeans(n_clusters=3, random_state=42, n_init=10)` on `X_aniso`.
4. Plot the result coloured by predicted label and overlay the centroids.

**Question to answer in a markdown cell below:** Compare your plot to the same data coloured by the **true** labels. Where does KMeans "cut" the wrong way, and why?


In [ ]:
# Exercise 3.2 — Your code here


---
## 4. Wrap a trained KMeans in a Gradio app

*From `lec_09c_gradio_app_huggingface_deploy.ipynb`*


### Exercise 4.1 [G8, *realistic*] — Wrap a trained KMeans in a Gradio app (local)

Re-use the four-feature model from Exercise 2.3 and wrap it in a small Gradio interface that runs locally (no Hugging Face deploy required for this exercise — that's the lecture-09c walk-through).

**Steps:**
1. Re-train (or pickle and reload) the four-feature `KMeans` model.
2. Write a function `predict_segment(genre, age, income, spending) -> str` that:
   - encodes `genre` as `0/1` exactly as in Exercise 2.3,
   - builds a single-row DataFrame with the four features in the same column order,
   - calls `model.predict(...)` and returns the cluster id (and, optionally, the human-readable label from your `cluster_labels` dict).
3. Build a `gr.Interface` (or `gr.Blocks`) with four input components — `gr.Radio` for genre, `gr.Slider` for age / income / spending — wired to your `predict_segment` function.
4. Call `.launch()` (the local URL `http://127.0.0.1:7860/` should open).

**Reminder:** the deploy-to-Hugging-Face step is in `lec_09c`. This exercise stops at "runs locally" — that is enough to demonstrate G8 mechanically.


In [ ]:
# Exercise 4.1 — Your code here


---
## 5. Optional / career-track stretch exercise

*From `lec_09d_kmeans_other_parameters.ipynb` — optional bucket*


### Exercise 5.1 [O1, *stretch — optional*] — `init='random'` vs `init='k-means++'`

KMeans is sensitive to where the centroids start. By default scikit-learn uses the smart initialisation `'k-means++'`, but you can ask for `'random'`.

**Steps:**
1. Use the **two-feature** mall-customers matrix (`Annual_Income_(k$)`, `Spending_Score`).
2. Fit two models with `K = 5`:
   - `KMeans(n_clusters=5, init="random", n_init=1, random_state=0)`
   - `KMeans(n_clusters=5, init="k-means++", n_init=1, random_state=0)`
3. Print `inertia_` for both. Which one is lower?
4. Repeat the experiment with `n_init=10`. Does the gap shrink?

**What this teaches:** why `n_init` exists and why `'k-means++'` is the default. Read `lec_09d_kmeans_other_parameters.ipynb` for the full discussion of the other constructor parameters.


In [ ]:
# Exercise 5.1 — Your code here


---
## Summary

After completing the **required** exercises (0.1, 1.1–1.3, 2.1–2.3, 3.1–3.2, 4.1) you will have practised every required learning goal G1–G8.

| Section | Required exercises | Skills practised | Goal(s) |
|---|---|---|---|
| 0 | 0.1 | One-sentence distinction between supervised and unsupervised | G1 |
| 1 | 1.1, 1.2, 1.3 | Loading data, fitting `KMeans`, reading `cluster_centers_` / `labels_` / `inertia_`, `predict()` on new points | G2, G3, G6 |
| 2 | 2.1, 2.2, 2.3 | Picking `K` with elbow + silhouette, profiling and labelling clusters | G4, G5 |
| 3 | 3.1, 3.2 | Recognising when KMeans assumptions break (non-convex, anisotropic) | G7 |
| 4 | 4.1 | Wrapping a trained model in a Gradio app | G8 |
| 5 *(optional)* | 5.1 | Tuning `init` and `n_init` | O1 |

**Key scikit-learn API patterns reinforced here:**

- `model.fit(X)` &nbsp;→&nbsp; train and store `cluster_centers_`, `labels_`, `inertia_`
- `model.predict(X_new)` &nbsp;→&nbsp; assign labels to **new** points
- `model.fit_predict(X)` &nbsp;→&nbsp; one-line shortcut for unsupervised learning
- `silhouette_score(X, labels)` &nbsp;→&nbsp; quality of a clustering
